# Options research

The workspace for option pricing models and option positions. Here you can:

1. load the Deribit option chains we record (or fetch one now),
2. look at the market's smile and fit the market surface (SVI),
3. calibrate pricing models to it and compare them in **implied-vol points**,
4. put each model through the validation gate,
5. read greeks, and design a book of perps, calls and puts by its **exposures** and **shock scenarios**,
6. write your own model and test it the same way.

Everything is in `src/options/` (pricing, the PDE model, calibration, surface, Deribit data) and
`src/portfolio/exposure.py`. Design notes and the review of the earlier notebook are in `docs/options_plan.md`.

**Build the dataset:** `python main.py --record-option-chains` (BTC and ETH every hour until Ctrl-C, about
150 KB per snapshot), or `python main.py --option-chain-snapshot` for a single snapshot.

## 0. Setup

In [ ]:
import os
import sys
from datetime import timedelta
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.options.calibration import Parameter, black76_per_expiry, calibrate, iv_errors, market_quotes
from src.options.deribit import fetch_chain, latest_chain, load_chains, save_chain
from src.options.pde import LocalVolJumpPDE
from src.options.pricing import Black76, MertonJump, greeks, implied_vol
from src.options.surface import fit_svi_surface
from src.options.validation import validate_model
from src.portfolio.exposure import MarketState, Position, exposures, scenario_grid

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Data

The latest stored snapshot, or a fresh one from Deribit's public API (it is saved too, so the dataset grows).
Prices are in USD (Deribit quotes in BTC; USD = BTC price x that expiry's forward). `forward` is each expiry's own
forward, and `t` is the time to expiry in years.

In [ ]:
CURRENCY = "BTC"

chain = latest_chain(CURRENCY)
if chain.empty:
    chain = fetch_chain(CURRENCY)
    print("saved", save_chain(chain))
snapshot = chain["timestamp"].iloc[0]
print(f"{CURRENCY} chain at {snapshot:%Y-%m-%d %H:%M} UTC: {len(chain)} contracts, {chain['expiry'].nunique()} expiries")
chain.groupby(chain["expiry"].dt.strftime("%Y-%m-%d")).agg(contracts=("strike", "size"), forward=("forward", "first"), open_interest=("open_interest", "sum"))

## 2. The market's smile

`market_quotes` keeps quotes with a bid and an ask (mid price), a spread under 30% of mid, and strikes between 0.5x and
2x the forward. It keeps only out-of-the-money options (puts below the forward, calls above), which are the liquid
side, and adds each quote's implied vol and vega.

Crypto smiles are usually lowest near the money and rise into both wings, with the put side higher (crash
protection costs more). The level changes with the expiry: that's the term structure.

In [ ]:
quotes = market_quotes(chain, max_relative_spread=0.3)
print(f"{len(quotes)} usable quotes")
expiries = sorted(quotes["t"].unique())
show = [expiries[min(i, len(expiries) - 1)] for i in (1, len(expiries) // 3, 2 * len(expiries) // 3, len(expiries) - 1)]
fig, ax = plt.subplots(figsize=(10, 5))
for t in dict.fromkeys(show):
    group = quotes[quotes["t"] == t]
    ax.plot(np.log(group["strike"] / group["forward"]), group["market_iv"] * 100, "o-", ms=3, label=f"{t * 365:.0f} days")
ax.set_xlabel("log-moneyness ln(K/F)"); ax.set_ylabel("implied vol (%)"); ax.legend(); ax.set_title("Market smile by expiry"); ax.grid(alpha=0.3)

## 3. The market surface (SVI)

Five numbers per expiry describe a smile (level, wing slope, skew, shift, curvature), and the expiries are joined
in time. This is what **risk** should use (greeks and scenarios consistent with market prices), and the baseline
any model has to be compared with. A good fit is well under 1 vol point.

In [ ]:
surface = fit_svi_surface(quotes)
svi_errors = iv_errors(surface, quotes)
rmse = lambda errors: float(np.sqrt(np.mean(errors ** 2)))
print(f"SVI surface: IV RMSE {rmse(svi_errors['iv_error_points']):.2f} vol points over {len(quotes)} quotes")
svi_errors.groupby(svi_errors["expiry"].dt.strftime("%Y-%m-%d"))["iv_error_points"].agg(rmse).round(2).to_frame("SVI IV RMSE (vol pts)")

## 4. Models against the market

Each model is fitted by `calibrate`, which minimises the price error divided by vega (about the error in vol
points). The table compares:

- **flat Black-76 per expiry:** one vol per expiry, no smile. The floor any model must beat.
- **Merton jumps:** one parameter set for all expiries. It can make a skew, but it can't bend to the term structure.
- **Local vol + jumps (PDE):** the earlier notebook's model, fixed (see `docs/options_plan.md`, section 6). It is
  slower, so it is fitted to one expiry here.

In [ ]:
flat = black76_per_expiry(quotes)
flat_errors = pd.concat([iv_errors(Black76(sigma), quotes[quotes["t"] == t]) for t, sigma in flat.items()])

merton = calibrate(lambda p: MertonJump(p["sigma"], p["intensity"], p["jump_mean"], p["jump_vol"]),
                   [Parameter("sigma", 0.4, 0.05, 1.5), Parameter("intensity", 0.5, 0.0, 5.0),
                    Parameter("jump_mean", -0.05, -0.5, 0.3), Parameter("jump_vol", 0.15, 0.02, 0.8)], quotes)
print(f"Merton: {merton.params} ({merton.evaluations} evaluations, {merton.seconds:.0f}s)")

one_expiry = min(expiries, key=lambda t: abs(t - 90 / 365))  # about three months
single = quotes[quotes["t"] == one_expiry]
anchor = float(single["forward"].iloc[0])
pde = calibrate(lambda p: LocalVolJumpPDE(sigma=p["sigma"], skew=p["skew"], intensity=p["intensity"], jump_mean=p["jump_mean"], jump_vol=0.2,
                                          anchor=anchor, grid_points=300, time_steps=100),
                [Parameter("sigma", 0.35, 0.05, 1.5), Parameter("skew", 0.05, 0.0, 0.8), Parameter("intensity", 0.3, 0.0, 3.0), Parameter("jump_mean", -0.05, -0.4, 0.1)],
                single, method="Nelder-Mead", maxiter=80)
print(f"PDE ({one_expiry * 365:.0f}-day expiry): {pde.params} ({pde.evaluations} evaluations, {pde.seconds:.0f}s)")

pd.DataFrame({
    "all expiries": {"SVI surface": rmse(svi_errors["iv_error_points"]), "flat Black-76 per expiry": rmse(flat_errors["iv_error_points"]), "Merton": merton.iv_rmse},
    f"{one_expiry * 365:.0f}-day expiry": {"SVI surface": rmse(svi_errors[svi_errors["t"] == one_expiry]["iv_error_points"]),
                                            "flat Black-76 per expiry": rmse(flat_errors[flat_errors["t"] == one_expiry]["iv_error_points"]),
                                            "Merton": rmse(merton.errors[merton.errors["t"] == one_expiry]["iv_error_points"]), "PDE local vol + jumps": pde.iv_rmse},
}).round(2).rename_axis("IV RMSE (vol points)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
k = np.log(single["strike"] / single["forward"])
ax.plot(k, single["market_iv"] * 100, "ko", ms=4, label="market")
for label, errors in (("SVI", svi_errors), ("Merton", merton.errors), ("PDE", pde.errors)):
    part = errors[errors["t"] == one_expiry]
    ax.plot(np.log(part["strike"] / part["forward"]), part["model_iv"] * 100, "-", label=label)
ax.set_xlabel("ln(K/F)"); ax.set_ylabel("implied vol (%)"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"{one_expiry * 365:.0f}-day smile: market vs models");

## 5. The gate

Every model must pass the arbitrage and consistency checks before it is used for anything: bounds, monotone and
convex in strike, put-call parity, deltas within bounds, calendar order. A failure says which check broke and
where.

In [ ]:
forward = float(quotes["forward"].median())
strikes = [forward * m for m in (0.5, 0.7, 0.85, 1.0, 1.15, 1.4, 1.8)]
gate_expiries = [7 / 365, 30 / 365, 90 / 365, 0.5]
rows = {}
for name, model in (("SVI surface", surface), ("Merton", merton.model), ("PDE", pde.model)):
    results = validate_model(model, forward=forward, strikes=strikes, expiries=gate_expiries if name != "PDE" else [one_expiry], tolerance=2e-4)
    rows[name] = {check.name: "pass" if check.passed else f"FAIL ({check.where})" for check in results}
pd.DataFrame(rows).T

## 6. Greeks

Per option contract (1 BTC on Deribit): delta (BTC per contract), gamma, vega (USD per vol point) and theta (USD per
day). Use the market surface for risk. Model greeks differ where the model's smile does.

In [ ]:
rows = []
for moneyness in (0.8, 0.9, 1.0, 1.1, 1.25):
    strike = round(forward * moneyness, -3)
    right = "put" if moneyness < 1 else "call"
    g = greeks(surface, forward, strike, one_expiry, right)
    rows.append({"strike": strike, "right": right, "price_usd": g.price, "iv": surface.iv(forward, strike, one_expiry), "delta": g.delta,
                 "gamma_per_1pct": g.gamma * (0.01 * forward) ** 2, "vega_per_vol_pt": g.vega * 0.01, "theta_per_day": g.theta / 365})
pd.DataFrame(rows).round(4)

## 7. Design a book by its exposures

A book of perps and options on the same coin is one exposure. The example is a **collar**: long BTC through the
perp, a protective put below, and a call sold above to pay for it. `exposures` gives delta, gamma, vega and theta
per coin by revaluing the whole book with the market surface. `scenario_grid` shows the P&L for large moves
(where greeks stop being enough), crossed with vol shocks.

Change the positions and watch the numbers: e.g. sell only the call (a covered call), or add a second put.

In [ ]:
now = pd.Timestamp(snapshot).to_pydatetime()
expiry = now + timedelta(days=one_expiry * 365)
spot = float(chain["index_price"].iloc[0])
market = MarketState(now=now, spot={"BTC": spot}, models={"BTC": surface})

book = [
    Position("perp", "BTC", 0.01, label="long perp"),
    Position("option", "BTC", 0.01, strike=round(spot * 0.85, -3), expiry=expiry, right="put", label="protective put"),
    Position("option", "BTC", -0.01, strike=round(spot * 1.2, -3), expiry=expiry, right="call", label="covered call"),
]
exposures(book, market).round(4)

In [ ]:
scenario_grid(book, market).round(0)

In [ ]:
naked = scenario_grid(book[:1], market)["+0 vol pts"]
collar = scenario_grid(book, market)["+0 vol pts"]
pd.DataFrame({"perp only": naked, "collar": collar}).plot(kind="bar", figsize=(10, 4), title="P&L by spot move (USD), vols unchanged").grid(alpha=0.3)

## 8. Your own model

A pricing model is any object with `name` and `price(forward, strike, t, right, discount=1.0)`, priced on the
forward and discounted. That's all calibration, the gate, greeks and the exposure layer need. The template below is
a Black-76 whose vol depends on moneyness (a toy "sticky" smile): replace its `price` with your model (a new PDE,
Heston, a neural pricer), give it a `sigma` (or a `shifted` method) so vega and vol scenarios work, then run it
through the same cells.

The toy fits the 3-month smile well, but the gate fails it in the far put wing: a quadratic smile rises so fast
there that a put's delta goes below -1, which no real option can do. That is what the gate is for: a model can fit
the quotes and still be unusable away from them.

In [ ]:
@dataclass(frozen=True)
class MySmileModel:
    sigma: float          # at-the-money vol
    skew: float = -0.2    # vol change per unit of ln(K/F)
    curvature: float = 0.5
    name: str = "my_smile"

    def price(self, forward, strike, t, right, discount=1.0):
        k = np.log(strike / forward)
        vol = max(self.sigma + self.skew * k + self.curvature * k * k, 0.01)
        return Black76(vol).price(forward, strike, t, right, discount)


mine = calibrate(lambda p: MySmileModel(p["sigma"], p["skew"], p["curvature"]),
                 [Parameter("sigma", 0.4, 0.05, 2.0), Parameter("skew", -0.1, -2.0, 2.0), Parameter("curvature", 0.3, 0.0, 5.0)], single)
print(f"my model on the {one_expiry * 365:.0f}-day expiry: IV RMSE {mine.iv_rmse:.2f} vol pts, params {mine.params}")
failed = [check for check in validate_model(mine.model, forward=forward, strikes=strikes, expiries=[one_expiry]) if not check.passed]
print("gate:", "all checks pass" if not failed else [(check.name, check.where) for check in failed])

## 9. History for backtests

`load_chains(currency, start=..., end=...)` returns every stored snapshot. Use it to test a model **out of sample**
(fit on one snapshot, price the next at its new forward) and to backtest option strategies at real bids and asks.
Those are steps 5 and 6 of the model gate in `docs/options_plan.md`.

In [ ]:
history = load_chains(CURRENCY)
print(f"{history['timestamp'].nunique() if not history.empty else 0} stored {CURRENCY} snapshots"
      + (f" from {history['timestamp'].min():%Y-%m-%d %H:%M} to {history['timestamp'].max():%Y-%m-%d %H:%M}" if not history.empty else ""))